## Another crack at random forest regression

In [3]:
import pandas as pd
import copy
import numpy as np

In [4]:
ogdf = pd.read_csv('./data/final_dataframes/main_training_dataframe.csv')
#I am now going to clean up some columns and add features
houroutdf=copy.deepcopy(ogdf)
houroutdf = houroutdf.drop(columns= ["Unnamed: 0"])
houroutdf['Month'] = houroutdf['MonthDay'].str[:2]
houroutdf['time_of_day'] = houroutdf['time'].str[10:]
# Convert to datetime and extract hour
houroutdf['time_of_day'] = houroutdf['time_of_day'].str[:3].astype(int)
houroutdf = houroutdf.drop(columns=['time'])
houroutdf = houroutdf[[col for col in houroutdf.columns if col != "Wind"] + ["Wind"]]
houroutdf = houroutdf.rename(columns={'Wind': 'Wind_Power'})
df0 = copy.deepcopy(houroutdf) #removing categorical variables for ease rn
df0 = df0.drop(columns=['MonthDay'])
df0

,temperature_2m_1,relative_humidity_2m_1,wind_speed_10m_1,wind_direction_10m_1,temperature_2m_33,relative_humidity_2m_33,wind_speed_10m_33,wind_direction_10m_33,temperature_2m_2,relative_humidity_2m_2,...,wind_speed_10m_34,wind_direction_10m_34,temperature_2m_38,relative_humidity_2m_38,wind_speed_10m_38,wind_direction_10m_38,Year,Month,time_of_day,Wind_Power
0,-7.4,81,17.9,146,-5.7,87,12.4,100,-13.4,85,...,13.0,132,-5.6,84,18.8,144,2019,01,0,1933.0
1,-7.0,80,18.9,145,-5.8,87,13.6,102,-13.6,87,...,14.6,130,-5.8,84,19.2,142,2019,01,1,1996.0
2,-7.1,80,21.3,146,-5.9,88,12.9,103,-14.1,89,...,16.0,121,-5.8,83,17.9,142,2019,01,2,1936.0
3,-6.8,80,22.2,148,-5.9,90,15.9,103,-12.8,86,...,16.2,122,-6.0,84,15.0,136,2019,01,3,2023.0
4,-6.8,81,22.4,143,-5.9,90,15.6,109,-11.9,88,...,16.1,114,-6.3,86,15.8,131,2019,01,4,2147.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35059,3.1,92,7.7,217,1.1,94,5.6,45,-2.5,99,...,4.9,234,1.8,98,4.6,198,2022,12,19,145.0
35060,2.6,97,2.5,188,1.0,95,4.6,51,-1.1,100,...,1.1,90,2.2,100,5.1,188,2022,12,20,118.0
35061,2.9,96,2.7,203,0.8,96,6.0,17,-0.9,99,...,1.8,90,3.4,99,2.2,189,2022,12,21,31.0
35062,3.8,94,3.1,54,0.6,96,5.2,335,3.0,90,...,3.3,96,3.0,100,1.8,127,2022,12,22,33.0


In [5]:
#now we need to take the entries in 'Wind_power' and then add n=24 of them together for power one day ahead
n=24
dayoutdf = copy.deepcopy(houroutdf)
dayoutdf['Wind_Power'] = dayoutdf['Wind_Power'].rolling(window=n).sum().shift(-n) #this replaces 'Wind' time t with the sum of the entries in 'Wind' at t+1,... t+24
df = copy.deepcopy(dayoutdf) #removing categorical variables for ease rn
df = df.drop(columns=['MonthDay'])
df = df.iloc[:-24]
df

,temperature_2m_1,relative_humidity_2m_1,wind_speed_10m_1,wind_direction_10m_1,temperature_2m_33,relative_humidity_2m_33,wind_speed_10m_33,wind_direction_10m_33,temperature_2m_2,relative_humidity_2m_2,...,wind_speed_10m_34,wind_direction_10m_34,temperature_2m_38,relative_humidity_2m_38,wind_speed_10m_38,wind_direction_10m_38,Year,Month,time_of_day,Wind_Power
0,-7.4,81,17.9,146,-5.7,87,12.4,100,-13.4,85,...,13.0,132,-5.6,84,18.8,144,2019,01,0,53069.0
1,-7.0,80,18.9,145,-5.8,87,13.6,102,-13.6,87,...,14.6,130,-5.8,84,19.2,142,2019,01,1,53355.0
2,-7.1,80,21.3,146,-5.9,88,12.9,103,-14.1,89,...,16.0,121,-5.8,83,17.9,142,2019,01,2,53663.0
3,-6.8,80,22.2,148,-5.9,90,15.9,103,-12.8,86,...,16.2,122,-6.0,84,15.0,136,2019,01,3,53689.0
4,-6.8,81,22.4,143,-5.9,90,15.6,109,-11.9,88,...,16.1,114,-6.3,86,15.8,131,2019,01,4,53600.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35035,7.1,83,20.2,202,7.0,83,17.9,214,-0.6,93,...,13.8,173,7.6,82,20.7,220,2022,12,19,36696.0
35036,7.8,81,21.1,205,6.9,83,15.0,204,-1.3,96,...,14.1,167,8.4,81,21.5,220,2022,12,20,34296.0
35037,6.6,83,16.1,219,5.8,85,10.8,195,-0.7,95,...,13.1,172,7.0,85,20.7,230,2022,12,21,31854.0
35038,7.6,82,19.6,221,5.5,86,9.4,198,-0.2,94,...,11.7,162,7.2,85,25.0,224,2022,12,22,29509.0


### Let's try with just wind data, over the course of one year

In [10]:
df = df.drop(columns=[col for col in df.columns if col.startswith("relative_humidity_2m_")])
df = df.drop(columns=[col for col in df.columns if col.startswith("wind_direction_10m_")])
df = df.drop(columns=[col for col in df.columns if col.startswith("temperature_2m_")])
df =df.drop(columns=["Year"])
df =df.drop(columns=["Month"])
df =df.drop(columns=["time_of_day"])



In [11]:
df

,wind_speed_10m_1,wind_speed_10m_33,wind_speed_10m_2,wind_speed_10m_8,wind_speed_10m_4,wind_speed_10m_23,wind_speed_10m_5,wind_speed_10m_39,wind_speed_10m_6,wind_speed_10m_7,...,wind_speed_10m_26,wind_speed_10m_27,wind_speed_10m_28,wind_speed_10m_29,wind_speed_10m_30,wind_speed_10m_32,wind_speed_10m_31,wind_speed_10m_34,wind_speed_10m_38,Wind_Power
0,17.9,12.4,11.4,11.7,18.0,22.7,14.3,13.9,12.2,11.9,...,14.0,22.0,11.7,11.7,11.2,9.9,14.0,13.0,18.8,53069.0
1,18.9,13.6,12.8,8.1,19.4,27.2,15.5,10.8,13.1,12.9,...,15.3,22.8,8.1,8.1,9.8,9.7,13.8,14.6,19.2,53355.0
2,21.3,12.9,13.6,6.8,18.9,29.2,16.1,12.3,13.4,14.6,...,21.1,24.7,6.8,6.8,10.2,9.2,14.0,16.0,17.9,53663.0
3,22.2,15.9,15.5,7.2,19.6,33.0,17.3,10.0,13.1,15.0,...,19.5,26.4,7.2,7.2,10.8,8.9,14.1,16.2,15.0,53689.0
4,22.4,15.6,16.2,9.7,15.9,32.5,16.6,13.0,15.0,16.7,...,18.9,26.2,9.7,9.7,13.8,10.8,15.3,16.1,15.8,53600.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35035,20.2,17.9,6.5,23.5,18.3,18.1,15.1,30.8,14.3,20.1,...,16.9,18.1,23.5,23.5,19.3,13.3,20.6,13.8,20.7,36696.0
35036,21.1,15.0,6.6,21.4,20.2,17.3,16.3,30.4,15.8,21.3,...,17.6,18.8,21.4,21.4,18.5,18.0,22.5,14.1,21.5,34296.0
35037,16.1,10.8,6.2,20.0,19.5,18.4,15.8,31.0,12.7,20.0,...,16.8,10.9,20.0,20.0,16.8,19.1,21.4,13.1,20.7,31854.0
35038,19.6,9.4,6.5,19.3,20.0,20.2,15.9,30.6,12.1,18.9,...,16.4,12.8,19.3,19.3,15.6,17.6,19.5,11.7,25.0,29509.0


In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from sklearn.feature_selection import RFE

In [30]:
df=df.head(8760)
X = df[[col for col in df.columns if col != "Wind_Power"]]
y = df["Wind_Power"]
# Step 1: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.01, random_state=614)

# Initialize and train Random Forest model
model = RandomForestRegressor(n_estimators=1000, max_depth=100, random_state=614)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate
mape = mean_absolute_percentage_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAPE: {mape:.4f}")
print(f"R² Score: {r2:.4f}")
print(model.feature_importances_)
print((pd.Series(model.feature_importances_)))

MAPE: 0.5453
R² Score: 0.6514
[0.02024673 0.02221028 0.01346315 0.0075602  0.01991951 0.0488726
 0.01074459 0.03091329 0.01072406 0.01203558 0.01549267 0.0094619
 0.010219   0.01589353 0.00555093 0.00551816 0.01069462 0.01595283
 0.1775144  0.01767526 0.01164935 0.03666443 0.06992734 0.01339339
 0.01878967 0.01707927 0.19368743 0.01076381 0.01184252 0.02088298
 0.01233419 0.01381198 0.00754097 0.00751743 0.01521554 0.0128548
 0.0115738  0.01620367 0.01760415]
0     0.020247
1     0.022210
2     0.013463
3     0.007560
4     0.019920
5     0.048873
6     0.010745
7     0.030913
8     0.010724
9     0.012036
10    0.015493
11    0.009462
12    0.010219
13    0.015894
14    0.005551
15    0.005518
16    0.010695
17    0.015953
18    0.177514
19    0.017675
20    0.011649
21    0.036664
22    0.069927
23    0.013393
24    0.018790
25    0.017079
26    0.193687
27    0.010764
28    0.011843
29    0.020883
30    0.012334
31    0.013812
32    0.007541
33    0.007517
34    0.015216
35    0.012

### Let's do PCA first

In [56]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [32]:
ogdf = pd.read_csv('./data/final_dataframes/main_training_dataframe.csv')
#I am now going to clean up some columns and add features
houroutdf=copy.deepcopy(ogdf)
houroutdf = houroutdf.drop(columns= ["Unnamed: 0"])
houroutdf['Month'] = houroutdf['MonthDay'].str[:2]
houroutdf['time_of_day'] = houroutdf['time'].str[10:]
# Convert to datetime and extract hour
houroutdf['time_of_day'] = houroutdf['time_of_day'].str[:3].astype(int)
houroutdf = houroutdf.drop(columns=['time'])
houroutdf = houroutdf[[col for col in houroutdf.columns if col != "Wind"] + ["Wind"]]
houroutdf = houroutdf.rename(columns={'Wind': 'Wind_Power'})
df0 = copy.deepcopy(houroutdf) #removing categorical variables for ease rn
df0 = df0.drop(columns=['MonthDay'])

In [35]:
#now we need to take the entries in 'Wind_power' and then add n=24 of them together for power one day ahead
n=24
dayoutdf = copy.deepcopy(houroutdf)
dayoutdf['Wind_Power'] = dayoutdf['Wind_Power'].rolling(window=n).sum().shift(-n) #this replaces 'Wind' time t with the sum of the entries in 'Wind' at t+1,... t+24
df = copy.deepcopy(dayoutdf) #removing categorical variables for ease rn
df = df.drop(columns=['MonthDay'])
df = df.iloc[:-24]
df =df.drop(columns=["Year"])
df =df.drop(columns=["Month"])
df =df.drop(columns=["time_of_day"])
df

,temperature_2m_1,relative_humidity_2m_1,wind_speed_10m_1,wind_direction_10m_1,temperature_2m_33,relative_humidity_2m_33,wind_speed_10m_33,wind_direction_10m_33,temperature_2m_2,relative_humidity_2m_2,...,wind_direction_10m_31,temperature_2m_34,relative_humidity_2m_34,wind_speed_10m_34,wind_direction_10m_34,temperature_2m_38,relative_humidity_2m_38,wind_speed_10m_38,wind_direction_10m_38,Wind_Power
0,-7.4,81,17.9,146,-5.7,87,12.4,100,-13.4,85,...,145,-11.1,88,13.0,132,-5.6,84,18.8,144,53069.0
1,-7.0,80,18.9,145,-5.8,87,13.6,102,-13.6,87,...,147,-10.4,87,14.6,130,-5.8,84,19.2,142,53355.0
2,-7.1,80,21.3,146,-5.9,88,12.9,103,-14.1,89,...,136,-10.9,89,16.0,121,-5.8,83,17.9,142,53663.0
3,-6.8,80,22.2,148,-5.9,90,15.9,103,-12.8,86,...,128,-10.0,87,16.2,122,-6.0,84,15.0,136,53689.0
4,-6.8,81,22.4,143,-5.9,90,15.6,109,-11.9,88,...,129,-9.4,86,16.1,114,-6.3,86,15.8,131,53600.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35035,7.1,83,20.2,202,7.0,83,17.9,214,-0.6,93,...,224,2.9,92,13.8,173,7.6,82,20.7,220,36696.0
35036,7.8,81,21.1,205,6.9,83,15.0,204,-1.3,96,...,221,2.4,94,14.1,167,8.4,81,21.5,220,34296.0
35037,6.6,83,16.1,219,5.8,85,10.8,195,-0.7,95,...,226,2.3,94,13.1,172,7.0,85,20.7,230,31854.0
35038,7.6,82,19.6,221,5.5,86,9.4,198,-0.2,94,...,232,2.4,94,11.7,162,7.2,85,25.0,224,29509.0


In [37]:
df = df.drop(columns=[col for col in df.columns if col.startswith("wind_direction_10m_")])

In [57]:
# Function from earlier
def apply_pca(df, target_column, n_components=None):
    features = df.drop(columns=[target_column])
    target = df[target_column]

    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    pca = PCA(n_components=n_components)
    features_pca = pca.fit_transform(features_scaled)

    pca_columns = [f'PC{i+1}' for i in range(features_pca.shape[1])]
    df_pca = pd.DataFrame(features_pca, columns=pca_columns, index=df.index)

    df_pca[target_column] = target.values

    return df_pca, pca

In [58]:
df_pca, pca = apply_pca(df, target_column='Wind_Power', n_components=30)

In [59]:
df_pca

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC22,PC23,PC24,PC25,PC26,PC27,PC28,PC29,PC30,Wind_Power
0,-6.333830,4.011712,0.215508,-0.991187,-2.098482,2.001079,-0.738391,-2.014317,0.837495,-0.227248,...,-0.211287,-0.451874,-0.252777,-0.075595,0.576519,0.936810,-0.443600,-0.396082,-0.248958,53069.0
1,-6.358420,3.960146,0.872456,-0.702470,-2.510346,2.428093,-0.014086,-3.096955,1.145792,-0.617094,...,-0.186709,-0.335097,-0.350688,0.093713,0.500001,0.528083,-0.326325,-0.497375,-0.389484,53355.0
2,-6.408366,4.074725,1.455812,-0.339035,-2.395591,2.019542,0.559328,-3.236090,1.131901,-0.948025,...,-0.365733,-0.248349,-0.559849,-0.211815,-0.410568,0.062512,-0.355720,-0.360525,-0.495212,53663.0
3,-6.305459,4.020099,1.794890,0.021511,-2.292855,1.955882,0.652098,-3.165267,1.032224,-0.999498,...,-0.501740,-0.307424,-0.287426,-0.307455,-1.155224,-0.239923,-0.227570,-0.848830,-0.387448,53689.0
4,-6.340751,3.899054,2.433781,0.214035,-2.096074,1.722104,0.380170,-2.448998,1.102263,-0.758326,...,-0.440587,-0.372391,-0.277229,0.318692,-1.587756,-0.445264,0.084738,-0.210691,-0.459634,53600.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35035,-0.849615,3.523683,5.376818,-0.812241,-0.584046,0.747277,-1.918854,0.370233,0.978887,1.507780,...,-0.049818,0.164354,-0.199785,-0.012029,-0.090599,-0.121934,-0.006607,-0.047751,0.215273,36696.0
35036,-1.030299,3.315370,6.059372,-0.112929,-0.775804,0.775619,-2.001645,-0.142167,1.310391,1.148467,...,-0.130526,-0.085173,0.101401,0.161266,0.092557,0.182545,0.149455,-0.404190,0.292278,34296.0
35037,-0.894127,3.863262,5.173427,0.103031,-0.593777,0.929996,-1.192078,0.104648,1.527387,1.206631,...,-0.200648,-0.024651,0.085779,0.074448,0.447711,0.522723,0.269220,-0.202927,-0.286952,31854.0
35038,-0.747461,3.775276,5.123445,0.355478,-0.780189,0.594600,-1.448093,-0.183103,1.471732,1.012153,...,-0.487652,-0.023856,0.029304,-0.254465,0.167292,0.103916,0.432299,-0.261391,-0.417377,29509.0


In [ ]:
df = df_pca

In [60]:
df=df.head(8760)
X = df[[col for col in df.columns if col != "Wind_Power"]]
y = df["Wind_Power"]
# Step 1: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.01, random_state=614)

# Initialize and train Random Forest model
model = RandomForestRegressor(n_estimators=1000, max_depth=100, random_state=614)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate
mape = mean_absolute_percentage_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAPE: {mape:.4f}")
print(f"R² Score: {r2:.4f}")
print(model.feature_importances_)
print((pd.Series(model.feature_importances_)))

MAPE: 0.3675
R² Score: 0.8179
[0.0028486  0.00290717 0.00840357 0.02132467 0.00604581 0.01376712
 0.00663895 0.00515278 0.00556166 0.00342539 0.00150509 0.00313072
 0.00346089 0.00323042 0.02182819 0.02933134 0.00631387 0.02980969
 0.00159413 0.0025424  0.00523288 0.03764912 0.01410757 0.01324783
 0.00164189 0.00238326 0.00411792 0.00314958 0.00296488 0.00420469
 0.00708063 0.00360747 0.00471542 0.00176082 0.00152271 0.00395112
 0.00129498 0.00159847 0.00369775 0.0024074  0.00396695 0.00590416
 0.00130227 0.00145495 0.00220018 0.00124868 0.00149993 0.00219827
 0.00127971 0.00205217 0.0042795  0.00291772 0.00268555 0.00595846
 0.00185475 0.00231385 0.15614518 0.00361156 0.00404457 0.00902071
 0.00352635 0.00329685 0.00448988 0.01137019 0.00508869 0.00653362
 0.01050157 0.00641552 0.05127643 0.0046861  0.00472377 0.00527195
 0.00225923 0.00382898 0.00949936 0.00423223 0.00558619 0.00840804
 0.00779055 0.00420642 0.18493083 0.00200691 0.00193999 0.0040583
 0.00163665 0.00195357 0.0052505 

### Scratch work

Try this out with all components at 6 and 10 PCA, should look at Mani's code for kNN, he has some nice plots.

In [ ]:
X = df[[col for col in df.columns if col != "Wind_Power"]]
y = df["Wind_Power"]
# Step 1: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.01, random_state=614)

# Step 2: Define base model
rf = RandomForestRegressor(n_estimators=100, random_state=42)

# Step 3: Use RFE to select top N features
n_features_to_select = 10  # Adjust as needed
rfe = RFE(estimator=rf, n_features_to_select=n_features_to_select)
rfe.fit(X_train, y_train)

# Step 4: Transform data
X_train_rfe = rfe.transform(X_train)
X_test_rfe = rfe.transform(X_test)

# Step 5: Train new model on selected features
rf_rfe = RandomForestRegressor(n_estimators=100, random_state=42)
rf_rfe.fit(X_train_rfe, y_train)
y_pred = rf_rfe.predict(X_test_rfe)

# Step 6: Evaluate
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAPE: {mean_absolute_percentage_error(y_test, y_pred):.4f}")

# Optional: View selected feature names
if hasattr(X, 'columns'):
    selected_features = X.columns[rfe.support_]
    print("Selected features:")
    print(selected_features)